### regridding CHIRPS data

### Average method

In [1]:
import os
import glob
import rasterio
from rasterio.enums import Resampling

# Input and output directories
input_dir = r"C:\Users\geri geri\Desktop\prepicitation riability\TM proposal\data\CHIRPS_Monthly_Abay\XGBOOST_corrected"
output_dir = os.path.join(input_dir, "resampled_0_25deg")

os.makedirs(output_dir, exist_ok=True)

# Resampling factor (0.05 → 0.25 = 5)
scale_factor = 5

# Get all tif files
tif_files = glob.glob(os.path.join(input_dir, "XGBOOST_corrected_*.tif"))

for tif in tif_files:
    with rasterio.open(tif) as src:
        # Calculate new shape
        new_height = src.height // scale_factor
        new_width = src.width // scale_factor

        # Update transform
        new_transform = src.transform * src.transform.scale(
            scale_factor,
            scale_factor
        )

        # Read and resample
        data = src.read(
            out_shape=(src.count, new_height, new_width),
            resampling=Resampling.average
        )

        # Update metadata
        profile = src.profile
        profile.update({
            "height": new_height,
            "width": new_width,
            "transform": new_transform
        })

        # Output filename
        filename = os.path.basename(tif)
        output_path = os.path.join(output_dir, filename.replace(".tif", "_0p25.tif"))

        # Write output
        with rasterio.open(output_path, "w", **profile) as dst:
            dst.write(data)

print("Resampling completed successfully!")

Resampling completed successfully!


### bilinear method

In [2]:
import os
import glob
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import Affine

# Input / output paths
input_dir = r"C:\Users\geri geri\Desktop\prepicitation riability\TM proposal\data\CHIRPS_Monthly_Abay\XGBOOST_corrected"
output_dir = os.path.join(input_dir, "resampled_0_25deg_bilinear")

os.makedirs(output_dir, exist_ok=True)

# Target resolution
target_res = 0.25

# Get files
tif_files = glob.glob(os.path.join(input_dir, "XGBOOST_corrected_*.tif"))

for tif in tif_files:
    with rasterio.open(tif) as src:

        # Original bounds (CRITICAL for preserving full extent)
        left, bottom, right, top = src.bounds

        # Compute exact new grid size (no truncation loss)
        new_width = int(np.round((right - left) / target_res))
        new_height = int(np.round((top - bottom) / target_res))

        # Build exact transform (same extent, new resolution)
        new_transform = Affine(
            target_res, 0, left,
            0, -target_res, top
        )

        # Read + resample using bilinear
        data = src.read(
            out_shape=(src.count, new_height, new_width),
            resampling=Resampling.bilinear
        )

        # Handle nodata properly (optional but recommended)
        if src.nodata is not None:
            data = np.where(data == src.nodata, np.nan, data)

        # Copy metadata
        profile = src.profile.copy()
        profile.update({
            "height": new_height,
            "width": new_width,
            "transform": new_transform,
            "driver": "GTiff"
        })

        out_name = os.path.basename(tif).replace(".tif", "_0p25_bilinear.tif")
        out_path = os.path.join(output_dir, out_name)

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data)

print("✅ Bilinear resampling completed with full extent preserved!")

✅ Bilinear resampling completed with full extent preserved!


In [3]:
import os
import glob
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import Affine

# Input / output paths
input_dir = r"C:\Users\geri geri\Desktop\prepicitation riability\TM proposal\data\CHIRPS_Monthly_Abay\XGBOOST_corrected"
output_dir = os.path.join(input_dir, "resampled_0_25deg_bilinear")

os.makedirs(output_dir, exist_ok=True)

# Target resolution
target_res = 0.25

# Get files
tif_files = glob.glob(os.path.join(input_dir, "XGBOOST_corrected_*.tif"))

for tif in tif_files:
    with rasterio.open(tif) as src:

        # Original bounds (CRITICAL for preserving full extent)
        left, bottom, right, top = src.bounds

        # Compute exact new grid size (no truncation loss)
        new_width = int(np.round((right - left) / target_res))
        new_height = int(np.round((top - bottom) / target_res))

        # Build exact transform (same extent, new resolution)
        new_transform = Affine(
            target_res, 0, left,
            0, -target_res, top
        )

        # Read + resample using bilinear
        data = src.read(
            out_shape=(src.count, new_height, new_width),
            resampling=Resampling.bilinear
        )

        # Handle nodata properly (optional but recommended)
        if src.nodata is not None:
            data = np.where(data == src.nodata, np.nan, data)

        # Copy metadata
        profile = src.profile.copy()
        profile.update({
            "height": new_height,
            "width": new_width,
            "transform": new_transform,
            "driver": "GTiff"
        })

        out_name = os.path.basename(tif).replace(".tif", "_0p25_bilinear.tif")
        out_path = os.path.join(output_dir, out_name)

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data)

print("✅ Bilinear resampling completed with full extent preserved!")

✅ Bilinear resampling completed with full extent preserved!
